# TF-IDF 분석

- 대응 코드: `tfidf.py`
- 빈도와 TF-IDF 상위어를 확인하고 원문 샘플을 점검.
- 표본 원문은 실행 화면에서만 확인.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

# 작업 폴더가 저장소 루트이거나 노트북 폴더여도 analysis/를 찾는다.
candidates = [Path.cwd(), *Path.cwd().parents]
analysis_root = next((candidate for base in candidates
                      for candidate in (base, base / "code" / "analysis", base / "ESG_comp" / "code" / "analysis")
                      if (candidate / "common" / "config.py").is_file()), None)
if analysis_root is None:
    raise FileNotFoundError("ESG_comp/code/analysis 폴더에서 노트북을 열어 주세요.")
if not os.environ.get("ESG_RESEARCH_ROOT"):
    raise RuntimeError("비공개 입력 위치를 ESG_RESEARCH_ROOT 환경변수로 먼저 지정하세요.")
sys.path.insert(0, str(analysis_root))
from common import config as cfg
from common.io_common import load_enriched


def show_csv(path, rows=10, columns=None):
    path = Path(path)
    if not path.is_file():
        print(f"저장 결과 없음: {path}\n입력 자료와 앞 단계를 확인한 뒤 실행하세요.")
        return None
    frame = pd.read_csv(path)
    display(frame[columns].head(rows) if columns else frame.head(rows))
    print(f"{path.name}: {len(frame):,}행")
    return frame


In [ ]:
LANG = "kr"
RUN_REPORT = False
RUN_RANK_CHECK = False
PREVIEW_ROWS = 10

In [ ]:
from stage2_frequency import tfidf as step
if LANG not in cfg.LANGS:
    raise ValueError(f"LANG은 {cfg.LANGS} 중 하나여야 합니다.")
merged, docs = load_enriched(LANG)
print(f"문서 {len(docs):,}건 · 선택된 언급 {len(merged):,}건")
display(merged.groupby("canonical").size().sort_values(ascending=False).head(PREVIEW_ROWS).rename("raw_count").to_frame())

In [ ]:
if RUN_REPORT:
    page_table = step.build_stage2_tfidf_report(LANG)
    display(page_table.head(PREVIEW_ROWS))
if RUN_RANK_CHECK:
    rank_table = step.build_rank_correlation_report()  # 한일 결과를 모두 사용
    display(rank_table.head(PREVIEW_ROWS))
if not (RUN_REPORT or RUN_RANK_CHECK):
    print("계산하려면 RUN_REPORT 또는 RUN_RANK_CHECK를 True로 바꾸세요.")

In [ ]:
out_dir = cfg.OUTPUT_DIR / LANG / "단계2_빈도분석"
for filename in (f"B_page_type별_TFIDF_{LANG}.csv", f"D_계열별_TFIDF_{LANG}.csv", f"F_ESG범주별_TFIDF_{LANG}.csv"):
    show_csv(out_dir / filename, PREVIEW_ROWS)
show_csv(cfg.OUTPUT_DIR / "significance" / "TFIDF_rank_vs_rawcount_spearman.csv", PREVIEW_ROWS)